# Stage 2: Bivariate analysis (two variables at a time)

**Status:** skeleton, 2026-10-23. Nothing here has been run beyond setup; it starts when Stage 1 is complete.

**Purpose.** Test how pairs of variables relate: whether they move together, in what shape (a straight line, a curve,
a step), how strongly, and for which groups. This turns Stage 1's descriptions into early signals.

**Two cautions.** With hundreds of thousands of rows almost any difference is "statistically significant", so the
size of an association matters more than its p-value. And two things moving together does not mean one causes the
other: a third factor (firm size, industry, city) often drives both. Stage 3 tests some of those third factors.

## How to read this notebook

- **Every observation is evidenced.** Observations are numbered (**B1**, **B2**, ...) and quote the numbers from
  the output cell directly above them. If a number is not on the page, it is not a finding.
- **Each observation is labelled with its strength of evidence**, as the research goal requires:
  *descriptive* (what the data shows), *observational* (an association that may have other explanations),
  *causal* (an effect, which needs a design this EDA does not have) or *projected* (an estimate of the future).
  Almost everything in the EDA is descriptive or observational.
- **Plain English first.** Technical terms are explained where they first appear and in [`GLOSSARY.md`](../../GLOSSARY.md).
- **Every chart names its source** underneath it.
- **Unit of analysis.** Each section works on one *lens*: a table with one row per thing being described. The lenses
  are listed below and defined in [`analysis/eda/build.py`](../eda/build.py); every column is described in
  [`analysis/eda/variables.yaml`](../eda/variables.yaml).

| Lens | One row per | Rows |
|---|---|---|
| `rights` | IP right (patent, trade mark, design, plant breeder's right) | 4,368,006 |
| `firms` | ABN that appears in any IP, R&D, procurement or tax source | 391,983 |
| `regions` | Postal area | 2,641 |
| `national` | Country, year and indicator | 127,309 |
| `timeline` | Month, for Australia (January 1904 to September 2026) | 1,473 |

**The measures used:**
- **Pearson r** (-1 to 1): how closely two variables follow a straight line.
- **Spearman rho** (-1 to 1): whether one rises (or falls) as the other rises, in any shape. When rho is much larger
  than r, the relationship is real but curved, or a few extreme values are distorting r.
- **LOWESS trend line**: a smooth line that follows the points without assuming a straight line.
- **Rates with 95% intervals**: for yes/no outcomes (for example, "holds a patent"), the share in each group, with the
  range the true share plausibly lies in.
- **Rank tests and effect sizes** (Mann-Whitney, Kruskal-Wallis): whether groups differ, and by how much, without
  letting a few extreme values decide.

## Setup

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from eda import bivariate as bi
from eda import data, load_lens, sql, style
from eda import multivariate as multi
from eda import univariate as uni

style.setup()

## 1. Screening: which pairs are worth a closer look?

For each lens, every pair of numeric variables is measured with r and rho (`bi.screen_pairs`) and the strongest
pairs, and those where rho and r disagree, are taken forward.

### 1.1 Screening the firms and places lenses

*Planned for Stage 1 (not yet run).* **Informs:** all. **Helpers:** `bi.screen_pairs`.

- [ ] `bi.screen_pairs` on the numeric columns of `firms` and `regions` (log-transformed where Stage 1 said so).
- [ ] Table of the strongest pairs and of pairs where rho exceeds r.

#### Observations
_To be written from the outputs above, each quoting its numbers._

## 2. IP rights

### 2.1 Time to registration by type, origin and attorney use

*Planned for Stage 1 (not yet run).* **Informs:** Q1, Q4. **Helpers:** `bi.binned_summary`, `bi.compare_groups`.

- [ ] Do Australian applicants, self-filers and organisations wait longer or shorter?

#### Observations
_To be written from the outputs above, each quoting its numbers._

### 2.2 How rights end, by who holds them

*Planned for Stage 1 (not yet run).* **Informs:** Q1, Q2. **Helpers:** `bi.rate_by`, `bi.rate_chart`.

- [ ] Share of rights that lapse early, by applicant origin, type and attorney use.

#### Observations
_To be written from the outputs above, each quoting its numbers._

## 3. Firms

### 3.1 Who uses the IP system

*Planned for Stage 1 (not yet run).* **Informs:** Q2. **Helpers:** `bi.rate_by`, `bi.rate_chart`.

- [ ] Share of firms holding a patent, trade mark or design by entity group, age band, remoteness and IRSAD decile.

#### Observations
_To be written from the outputs above, each quoting its numbers._

### 3.2 IP and R&D

*Planned for Stage 1 (not yet run).* **Informs:** Q3, Q4. **Helpers:** `bi.scatter_trend(logx, logy)`, `bi.rate_by`.

- [ ] R&D spend against patent applications; share of R&D Tax Incentive claimants that have never filed IP (the 'missing filers'), by group.

#### Observations
_To be written from the outputs above, each quoting its numbers._

### 3.3 IP and government contracts, and income

*Planned for Stage 1 (not yet run).* **Informs:** Q3. **Helpers:** `bi.compare_groups`.

- [ ] IP holders' share of AusTender contracts and value; income of large firms with and without IP.

#### Observations
_To be written from the outputs above, each quoting its numbers._

## 4. Places

### 4.1 IP per 1,000 businesses against advantage and remoteness

*Planned for Stage 1 (not yet run).* **Informs:** Q2, Q4. **Helpers:** `bi.scatter_trend`, `bi.binned_summary`.

- [ ] Rates against IRSAD decile, IEO and IER scores, median income and remoteness.
- [ ] Which areas file far more or less than their advantage would suggest.

#### Observations
_To be written from the outputs above, each quoting its numbers._

### 4.2 R&D claimants against IP filing

*Planned for Stage 1 (not yet run).* **Informs:** Q4. **Helpers:** `bi.scatter_trend`.

- [ ] Places with many R&D claimants but little IP filing.

#### Observations
_To be written from the outputs above, each quoting its numbers._

## 5. Australia and other countries

### 5.1 Australia against comparators, indicator by indicator

*Planned for Stage 1 (not yet run).* **Informs:** Q1. **Helpers:** `bi.scatter_trend`, `bi.pair_stats`.

- [ ] Pairs such as R&D spending against resident patenting, IP royalties paid against received, across OECD countries, with Australia marked.
- [ ] Caution: series that all trend upward correlate whatever their link; compare levels in a year, or changes.

#### Observations
_To be written from the outputs above, each quoting its numbers._

## 6. Australia over time

Comparisons over time are where misleading correlations are most likely: two series that both rise over decades will
correlate whatever their connection. So this section compares *changes*, not levels: the months just before and after
a policy change, and changes from one year to the next.

### 6.1 Before and after each policy change

*Planned for Stage 2 (not yet run).* **Informs:** Q4, Q5. **Helpers:** `bi.binned_summary`, `bi.compare_groups`, `uni.monthly`.

- [ ] For each change, the level of the series it targets in the 12 and 24 months before against after, beside a series it should not affect (for example, trade marks for a patent change).
- [ ] Rushes to file before a change (April 2013, August 2021) are separated from lasting shifts after it.

#### Observations
_To be written from the outputs above, each quoting its numbers._

### 6.2 From application to outcome

*Planned for Stage 2 (not yet run).* **Informs:** Q1. **Helpers:** `bi.pair_stats` on year-on-year changes.

- [ ] How the gap between applications and grants or registrations has moved over time (counts a set number of months apart, and `days_to_registration` from the rights lens by year).

#### Observations
_To be written from the outputs above, each quoting its numbers._

### 6.3 Australian against overseas applicants

*Planned for Stage 2 (not yet run).* **Informs:** Q2. **Helpers:** `bi.pair_stats`, `bi.scatter_trend`.

- [ ] Whether Australian and overseas applications move together from year to year, and where they part.

#### Observations
_To be written from the outputs above, each quoting its numbers._

## 7. Summary of Stage 2

| # | Pair | What was found | Evidence | Strength | Carried to Stage 3 |
|---|---|---|---|---|---|
| | | _To be written_ | | | |